# PyTorch Fundamentals

## Overview

PyTorch is the dominant deep learning framework for research and production. Its define-by-run (dynamic) computation graph makes debugging natural — code executes like regular Python.

**Core abstractions:**

| Concept | Role |
|---|---|
| `Tensor` | N-dimensional array on CPU or GPU |
| `autograd` | Automatic differentiation via computation graph |
| `nn.Module` | Base class for all neural network layers and models |
| `DataLoader` | Batched, shuffled data iteration |
| `Optimizer` | Updates parameters using computed gradients |

**Training loop:**
```
for batch in dataloader:
    optimizer.zero_grad()   # clear previous gradients
    output = model(batch)   # forward pass
    loss = criterion(output, target)
    loss.backward()         # backward pass: compute gradients
    optimizer.step()        # update parameters
```

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
rng = np.random.default_rng(42)
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Simulate ecological tabular data
n = 800
elevation   = rng.uniform(50, 400, n).astype(np.float32)
nitrate     = rng.gamma(2, 2, n).astype(np.float32)
phosphorus  = rng.gamma(1.5, 1.5, n).astype(np.float32)
treatment   = rng.choice([0., 1.], n).astype(np.float32)
richness    = (25 - 0.04*elevation - 0.8*nitrate + 2.5*treatment
               + rng.normal(0, 3, n)).clip(0).astype(np.float32)
X_np = np.column_stack([elevation, nitrate, phosphorus, treatment])
y_np = richness
print(f"\nData: X={X_np.shape}, y={y_np.shape}")

---
## Tensors and Autograd

In [ ]:
# Tensors: the fundamental data structure
X = torch.from_numpy(X_np)
y = torch.from_numpy(y_np).unsqueeze(1)   # (n,) -> (n,1)
print(f"X dtype: {X.dtype}, shape: {X.shape}")
print(f"y dtype: {y.dtype}, shape: {y.shape}")

# Autograd: automatic differentiation
w = torch.tensor([1.0, -0.5, 0.2, 1.0], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
# Manual linear prediction + MSE
y_pred = X @ w + b
loss   = ((y_pred.unsqueeze(1) - y) ** 2).mean()
loss.backward()   # compute gradients
print(f"\nManual forward pass:")
print(f"  Loss:    {loss.item():.4f}")
print(f"  dL/dw:   {w.grad[:3].detach().numpy().round(4)}")
print(f"  dL/db:   {b.grad.item():.4f}")
print("autograd fills .grad attributes automatically on .backward()")

---
## Building a Neural Network with nn.Module

In [ ]:
class FeedforwardNet(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout=0.2):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = FeedforwardNet(input_dim=4, hidden_dims=[64, 32], dropout=0.2).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

---
## Training Loop

In [ ]:
# Train/val split -> TensorDataset -> DataLoader
split = int(0.8 * n)
X_tr, X_va = X[:split].to(device), X[split:].to(device)
y_tr, y_va = y[:split].to(device), y[split:].to(device)

# Normalise inputs using training stats
X_mean, X_std = X_tr.mean(0), X_tr.std(0)
X_tr_n = (X_tr - X_mean) / (X_std + 1e-8)
X_va_n = (X_va - X_mean) / (X_std + 1e-8)

train_loader = DataLoader(TensorDataset(X_tr_n, y_tr),
                           batch_size=64, shuffle=True)

model    = FeedforwardNet(4, [64, 32], dropout=0.2).to(device)
optimiser = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, patience=10, factor=0.5)

train_losses, val_losses = [], []
for epoch in range(150):
    model.train()
    batch_losses = []
    for Xb, yb in train_loader:
        optimiser.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimiser.step()
        batch_losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_va_n), y_va).item()
    train_losses.append(np.mean(batch_losses))
    val_losses.append(val_loss)
    scheduler.step(val_loss)

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(train_losses, label='Train loss', color='steelblue')
ax.plot(val_losses,   label='Val loss',   color='#e74c3c')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss')
ax.set_title('Training Curve')
ax.legend(); plt.tight_layout(); plt.show()
final_rmse = np.sqrt(val_losses[-1])
print(f"Final val RMSE: {final_rmse:.3f}")

In [ ]:
# Saving and loading
import tempfile, os
model.eval()
with torch.no_grad():
    y_pred_val = model(X_va_n).cpu().numpy().ravel()
    y_true_val = y_va.cpu().numpy().ravel()

r2 = 1 - np.sum((y_pred_val - y_true_val)**2) / np.sum((y_true_val - y_true_val.mean())**2)
print(f"Validation R2:   {r2:.3f}")
print(f"Validation RMSE: {np.sqrt(np.mean((y_pred_val-y_true_val)**2)):.3f}")

# Save checkpoint
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, 'model.pt')
    torch.save({
        'model_state_dict':     model.state_dict(),
        'optimiser_state_dict': optimiser.state_dict(),
        'epoch':                150,
        'val_loss':             val_losses[-1],
        'X_mean': X_mean, 'X_std': X_std,
    }, path)
    ckpt = torch.load(path, map_location='cpu')
print(f"Checkpoint keys: {list(ckpt.keys())}")
print("Always save X_mean/X_std alongside model weights for correct inference")

---

## Common Pitfalls

**1. Forgetting `optimizer.zero_grad()` before `loss.backward()`**  
PyTorch accumulates gradients by default — each call to `.backward()` adds to existing `.grad` values. Without `zero_grad()`, gradients from previous batches contaminate the current update, producing incorrect parameter updates that are difficult to diagnose.

**2. Not switching between `model.train()` and `model.eval()`**  
`model.train()` enables dropout and batch normalisation training behaviour; `model.eval()` disables them for inference. Evaluating in training mode produces noisier, non-reproducible metrics. Forgetting `eval()` before validation inflates loss variance.

**3. Normalising with test-set statistics**  
Computing `X_mean` and `X_std` from the full dataset including the validation/test portion leaks their distribution into training. Always compute normalisation statistics on the training set only and apply the same values to all other splits.

**4. Using `loss.item()` for logging but `.backward()` on a detached tensor**  
Calling `.item()` converts a tensor to a Python scalar and detaches it from the computation graph — if done before `.backward()`, the gradient information is lost. Always call `.backward()` on the loss tensor before converting to scalar for logging.

**5. Not using `torch.no_grad()` during evaluation**  
Without `torch.no_grad()`, PyTorch builds and stores a computation graph during forward passes — doubling memory usage and slowing inference. Always wrap validation and inference code in `with torch.no_grad():`.
---
*python_methods_library - Samantha McGarrigle*